# FTMO Forex Data Download
Downloads 2 years of 15-min forex data from Dukascopy for 7 pairs.

**Pairs:** GBPUSD, GBPJPY, EURJPY, USDCAD, EURCHF, AUDNZD, EURGBP

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/ftmo_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
import struct
import lzma
import time
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta

import pandas as pd
import requests
from requests.adapters import HTTPAdapter

DUKASCOPY_URL = 'https://datafeed.dukascopy.com/datafeed'

POINT_VALUES = {
    'EURUSD': 1e5, 'GBPUSD': 1e5, 'GBPJPY': 1e3, 'EURJPY': 1e3,
    'USDCAD': 1e5, 'EURCHF': 1e5, 'AUDNZD': 1e5, 'EURGBP': 1e5,
}

PAIRS = ['GBPUSD', 'GBPJPY', 'EURJPY', 'USDCAD', 'EURCHF', 'AUDNZD', 'EURGBP']
START_DATE = '2024-01-01'
END_DATE = '2025-12-31'
TIMEFRAME = 15  # minutes
MAX_WORKERS = 12


def make_session():
    s = requests.Session()
    s.mount('https://', HTTPAdapter(pool_maxsize=MAX_WORKERS))
    s.headers.update({
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
        'Accept': '*/*',
        'Accept-Encoding': 'gzip, deflate',
    })
    return s


def download_hour(session, pair, dt, retries=3):
    month_zero = dt.month - 1
    url = f'{DUKASCOPY_URL}/{pair}/{dt.year}/{month_zero:02d}/{dt.day:02d}/{dt.hour:02d}h_ticks.bi5'
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=15)
            if resp.status_code == 200 and len(resp.content) > 0:
                return dt, resp.content
            if resp.status_code == 404:
                return dt, None
            if resp.status_code in (429, 500, 502, 503):
                time.sleep(2 * (attempt + 1))
                continue
        except requests.RequestException:
            time.sleep(1)
    return dt, None


def parse_bi5(data, pair, hour_dt):
    try:
        raw = lzma.decompress(data)
    except lzma.LZMAError:
        return []
    point_value = POINT_VALUES.get(pair, 1e5)
    ticks = []
    row_size = 20
    for i in range(0, len(raw), row_size):
        if i + row_size > len(raw):
            break
        ms_offset, ask_raw, bid_raw, ask_vol, bid_vol = struct.unpack('>IIIff', raw[i:i+row_size])
        timestamp = hour_dt + timedelta(milliseconds=ms_offset)
        mid = ((ask_raw + bid_raw) / 2) / point_value
        ticks.append({'datetime': timestamp, 'mid': mid, 'vol': ask_vol + bid_vol})
    return ticks


def resample_ticks(ticks, timeframe_minutes):
    """Convert raw ticks to OHLCV bars. Returns list of dicts."""
    if not ticks:
        return []
    df = pd.DataFrame(ticks)
    df.set_index('datetime', inplace=True)
    df.sort_index(inplace=True)
    rule = f'{timeframe_minutes}min'
    ohlcv = df['mid'].resample(rule).agg(open='first', high='max', low='min', close='last')
    ohlcv['volume'] = df['vol'].resample(rule).sum()
    ohlcv.dropna(subset=['open'], inplace=True)
    ohlcv.index.name = 'datetime'
    return ohlcv.reset_index().to_dict('records')


def download_pair(pair, start_date, end_date, timeframe_minutes, output_dir, max_workers=12):
    output_file = os.path.join(output_dir, f'{pair}_{timeframe_minutes}m.csv')

    if os.path.exists(output_file):
        existing = pd.read_csv(output_file)
        if len(existing) > 30000:
            print(f'  {pair}: Already exists ({len(existing)} bars). Skipping.')
            return output_file
        else:
            print(f'  {pair}: Exists but only {len(existing)} bars. Re-downloading.')

    start = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')

    hours = []
    current = start
    while current < end:
        if current.weekday() < 5:
            hours.append(current)
        current += timedelta(hours=1)

    total = len(hours)
    print(f'  {pair}: Downloading {total} hours ({start_date} to {end_date})...')

    completed = 0
    data_hours = 0
    total_bars = 0
    batch_size = 100
    session = make_session()
    start_time = time.time()
    header_written = False

    # Write bars to CSV incrementally — never hold all ticks in memory
    with open(output_file, 'w', newline='') as csvfile:
        writer = None

        for batch_start in range(0, total, batch_size):
            batch = hours[batch_start:batch_start + batch_size]
            batch_ticks = []

            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = {executor.submit(download_hour, session, pair, dt): dt for dt in batch}
                for future in as_completed(futures):
                    dt, data = future.result()
                    completed += 1
                    if data:
                        ticks = parse_bi5(data, pair, dt)
                        batch_ticks.extend(ticks)
                        data_hours += 1

            # Resample this batch's ticks to bars and write to CSV
            bars = resample_ticks(batch_ticks, timeframe_minutes)
            if bars:
                if writer is None:
                    writer = csv.DictWriter(csvfile, fieldnames=bars[0].keys())
                    writer.writeheader()
                writer.writerows(bars)
                csvfile.flush()
                total_bars += len(bars)

            # batch_ticks is now garbage collected — no memory buildup

            if completed % 500 == 0 or batch_start + batch_size >= total:
                pct = completed / total * 100
                elapsed = time.time() - start_time
                rate = completed / elapsed if elapsed > 0 else 0
                eta = (total - completed) / rate / 60 if rate > 0 else 0
                print(f'    {pair}: {pct:.0f}% ({completed}/{total}, {data_hours} hrs, {total_bars} bars, ETA: {eta:.0f}min)')

    if total_bars == 0:
        print(f'  {pair}: No data downloaded!')
        os.remove(output_file)
        return ''

    # Final step: sort the CSV (batches arrive out of order from threading)
    print(f'  {pair}: Sorting {total_bars} bars...')
    df = pd.read_csv(output_file, parse_dates=['datetime'])
    df.sort_values('datetime', inplace=True)
    df.drop_duplicates(subset=['datetime'], inplace=True)
    df.to_csv(output_file, index=False)
    print(f'  {pair}: Done — {len(df)} bars saved to {output_file}')
    return output_file

print('Functions loaded. Memory-efficient: ticks are resampled and flushed per batch.')

In [ ]:
# Download all 7 pairs sequentially with 5 workers each
# Each pair saves to Google Drive immediately, so if session dies you keep what's done

for pair in PAIRS:
    print(f'\n{"="*50}')
    print(f'Starting {pair}')
    print(f'{"="*50}')
    download_pair(pair, START_DATE, END_DATE, TIMEFRAME, OUTPUT_DIR, MAX_WORKERS)

print('\n\n===== ALL DONE =====')
!ls -lh {OUTPUT_DIR}/*.csv

In [ ]:
# Verify data quality
print(f'{"Pair":<10} {"Bars":>8} {"First Date":<22} {"Last Date":<22} {"Coverage"}')
print('-' * 80)
for pair in PAIRS:
    f = os.path.join(OUTPUT_DIR, f'{pair}_{TIMEFRAME}m.csv')
    if os.path.exists(f):
        df = pd.read_csv(f, parse_dates=['datetime'])
        bars = len(df)
        first = df['datetime'].min()
        last = df['datetime'].max()
        # Expected ~48K bars for 2 years of 15min data
        coverage = f'{bars/48000*100:.0f}%'
        print(f'{pair:<10} {bars:>8} {str(first):<22} {str(last):<22} {coverage}')
    else:
        print(f'{pair:<10} MISSING')

In [ ]:
# Download the CSVs as a zip (alternative to Google Drive)
import shutil
shutil.make_archive('/content/ftmo_data', 'zip', OUTPUT_DIR)

from google.colab import files
files.download('/content/ftmo_data.zip')
print('Download started. Check your browser downloads.')